In [ ]:
import os

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

pd.options.mode.copy_on_write = True

In [ ]:
###%matplotlib widget

# Constants

In [ ]:
eyelink_freq = 1000
markers_freq = 120
marker_count = 1
EYELINK_MISSING = -32768

# Load data

In [ ]:
session_dir = os.path.join(
    os.environ["ROS_BAG_DIR"], "session_2025-08-12_15-19-30"
)
# loaded_dfs = rosbag_session_to_dfs(
#     session_dir, topics=["/eyelink/sample", "/markers"], save=True
# )
# eyelink_loaded_df = loaded_dfs["/eyelink/sample"]
# markers_loaded_df = loaded_dfs["/markers"]

eyelink_csv_path = os.path.join(session_dir, "eyelink_sample.csv")
eyelink_received_csv_path = os.path.join(
    session_dir, "eyelink_received", "last.csv"
)
markers_csv_path = os.path.join(session_dir, "markers.csv")

eyelink_df = pd.read_csv(eyelink_csv_path)
eyelink_received_df = pd.read_csv(eyelink_received_csv_path)
markers_df = pd.read_csv(markers_csv_path)


In [ ]:
eyelink_df

In [ ]:
eyelink_received_df

In [ ]:
markers_df

# Format timestamps to seconds and check for monotonicity

In [ ]:
eyelink_df["bag_time"] = eyelink_df["bag_time_ns"] / 1e9
markers_df["bag_time"] = markers_df["bag_time_ns"] / 1e9

eyelink_df["time"] = (
    eyelink_df["header.stamp.sec"] + eyelink_df["header.stamp.nanosec"] / 1e9
)
markers_df["time"] = (
    markers_df["header.stamp.sec"] + markers_df["header.stamp.nanosec"] / 1e9
)
markers_df["original_time"] = (
    markers_df["header_original.stamp.sec"]
    + markers_df["header_original.stamp.nanosec"] / 1e9
)

eyelink_df["eyelink_time"] = eyelink_df["eyelink_time_ms"] / 1e3

eyelink_times = eyelink_df[["bag_time", "time", "eyelink_time"]]
markers_times = markers_df[["bag_time", "time", "original_time"]]

assert markers_df["frame_number"].is_monotonic_increasing
for df in [eyelink_times, markers_times]:
    for col in df.columns:
        assert df[col].is_monotonic_increasing, (
            f"{col} is not monotonic increasing for "
        )

# Verify timestamps integrity

In [ ]:
d = markers_times.time - markers_times.original_time
d = pd.DataFrame(d)
print("Time shift between markers adjusted and original time")
d.describe().round(5)

In [ ]:
print(f"Expected time diff: {1 / eyelink_freq:.3f}")
eyelink_times.diff().describe().round(4)

In [ ]:
print(f"Expected time diff: {1 / markers_freq:.4f}")
markers_times.diff().describe().round(4)

In [ ]:
d = eyelink_times.time.diff() - eyelink_times.eyelink_time.diff()
d = pd.DataFrame(d)
print("Error between eyelink and bag time diffs")
d.describe().round(5)

# Select relevant columns, standardize time, and drop invalid rows

In [ ]:
edf = eyelink_df.copy(deep=True)
mdf = markers_df.copy(deep=True)

edf = edf[
    [
        "time",
        "left_x",
        "left_y",
        "left_pupil",
        "right_x",
        "right_y",
        "right_pupil",
        "input",
    ]
]
mdf = mdf[
    [
        "time",
        "marker_0_x",
        "marker_0_y",
        "marker_0_z",
    ]
]

min_time = min(edf.time.min(), mdf.time.min())
edf["time"] = edf["time"] - min_time
mdf["time"] = mdf["time"] - min_time

edf["time_diff"] = edf["time"].diff()
mdf["time_diff"] = mdf["time"].diff()

for col in [
    "left_x",
    "left_y",
    "left_pupil",
    "right_x",
    "right_y",
    "right_pupil",
]:
    assert (edf[col] - edf[col].astype(int)).sum() == 0
    edf[col] = edf[col].astype(int).replace(EYELINK_MISSING, np.nan)

edf_drop = edf.dropna(
    subset=[
        "left_x",
        "left_y",
        "left_pupil",
        "right_x",
        "right_y",
        "right_pupil",
    ]
)
mdf_drop = mdf.dropna(subset=["marker_0_x", "marker_0_y", "marker_0_z"])
mdf_drop["time_markers"] = mdf.time

edf

# Merge eyelink and marker data on closest time

In [ ]:
merged = pd.merge_asof(
    edf_drop,
    mdf_drop,
    on="time",
    direction="backward",
    tolerance=edf.time_diff.max(),
    suffixes=("_eyelink", "_markers"),
)
assert merged.shape[0] == edf_drop.shape[0]
matched = merged.time_markers.notna().sum()
unique = (merged.time_markers.dropna() * 1000).astype(int).unique()

print(
    f"Matched {matched.sum()} ({unique.shape[0]} unique) out of {mdf.shape[0]} marker data points"
)

merged.head(20)

# Remove duplicated marker data as a result of the merge

In [ ]:
no_dups = merged.copy(deep=True)
marker_cols = [col for col in no_dups.columns if "marker" in col]
merged_markers = no_dups[marker_cols]
diff = merged_markers.time_markers.diff()
merged_markers[diff.notna() & (diff < 1e-6)] = np.nan
no_dups[marker_cols] = merged_markers

new_matched = no_dups.time_markers.notna()
new_unique = (no_dups.time_markers.dropna() * 1000).astype(int).unique()
assert unique.shape[0] == new_unique.shape[0], (
    f"Expected {unique.shape[0]} unique marker data points, got {new_unique.shape[0]}"
)
assert new_matched.sum() == new_unique.shape[0], (
    f"Expected all {new_matched.sum()} matched marker data points to be unique, got {new_unique.shape[0]} unique"
)

print(
    f"After removing duplicates, {new_matched.sum()} out of previous {matched.sum()} marker data points remain"
)
no_dups.head(20)

# Remove rows where the gap between marker datapoints is too large to be interpolated accurately

In [ ]:
print(
    f"Marker time gaps before merging | "
    f"min: {no_dups.time_diff_markers.min():.4f}, "
    f"max: {no_dups.time_diff_markers.max():.4f}"
)
gaps = no_dups.time[no_dups.time_markers.notna()].diff()
print(
    f"Marker time gaps after merging | "
    f"min: {gaps.min():.4f}, "
    f"max: {gaps.max():.4f}"
)

In [ ]:
asof = no_dups.asof(
    no_dups.index, subset=["marker_0_x", "marker_0_y", "marker_0_z"]
)
to_drop = asof.time[
    asof.time.isna()
    | (no_dups.time > asof.time + no_dups.time_diff_markers.bfill())
]
drop_times = to_drop.unique()
no_gaps = no_dups[~asof.time.isin(drop_times)]

print(
    f"Found {to_drop.shape[0]} rows (forming {len(drop_times)} contiguous gaps) where "
    f"the time since the last marker datapoint is greater than the maximum expected gap: "
    f"{no_dups.time_diff_markers.max():.4f}"
)
num_dropped = no_dups.shape[0] - no_gaps.shape[0]
print(f"Dropped {num_dropped} out of {no_dups.shape[0]} rows")
no_gaps

# Interpolate marker data onto eyelink time points

In [ ]:
interp = no_gaps.copy(deep=True)
interp.set_index("time", inplace=True)
markers_cols = ["marker_0_x", "marker_0_y", "marker_0_z"]
interp[markers_cols] = interp[markers_cols].interpolate(method="slinear")
interp.reset_index(inplace=True)
interp

# Clean up data for training

In [ ]:
df = interp.copy(deep=True)
df = df.drop(
    columns=[
        "left_pupil",
        "right_pupil",
        "input",
        "time_diff_eyelink",
        "time_diff_markers",
        "time_markers",
    ]
)
df.to_csv(os.path.join(session_dir, "calibration_data.csv"), index=False)

In [ ]:
X = torch.tensor(df[["left_x", "left_y", "right_x", "right_y"]].values)
Y = torch.tensor(df[["marker_0_x", "marker_0_y", "marker_0_z"]].values)

torch.save(X, os.path.join(session_dir, "eyelink.pt"))
torch.save(Y, os.path.join(session_dir, "markers.pt"))

# Plotting functions

In [ ]:
def plot_eyelink_markers(edf, mdf):
    fig, ax = plt.subplots(7, 1, sharex=True, figsize=(10, 15))
    i = 0
    for col in ["left_x", "left_y", "right_x", "right_y"]:
        ax[i].plot(edf.time, edf[col])
        ax[i].set_title(f"Eyelink {col}")
        i += 1

    for col in ["marker_0_x", "marker_0_y", "marker_0_z"]:
        ax[i].plot(mdf.time, mdf[col])
        ax[i].set_title(f"Markers {col}")
        i += 1

    return fig


plot_eyelink_markers(edf, mdf)
plt.savefig(os.path.join(session_dir, "eyelink_markers.png"))
plt.show()
plot_eyelink_markers(interp, interp)
plt.savefig(os.path.join(session_dir, "eyelink_markers_interp.png"))
plt.show()

In [ ]:
def animate_eyelink(df, fr=10, save_path="eyelink.gif"):
    freq = 1 / df.time.diff().median()
    interval = 1 / fr

    fig, ax = plt.subplots()
    (points,) = ax.plot([], [], "o")
    ax.set_xlim(-10000, 10000)
    ax.set_ylim(-15000, 15000)

    def init():
        row = df.iloc[0]
        left_x, left_y, right_x, right_y = row[
            ["left_x", "left_y", "right_x", "right_y"]
        ]
        points.set_data([left_x, right_x], [left_y, right_y])
        return (points,)

    def animate(i):
        row = df.iloc[int(i * interval * freq)]
        left_x, left_y, right_x, right_y = row[
            ["left_x", "left_y", "right_x", "right_y"]
        ]
        points.set_data([left_x, right_x], [left_y, right_y])
        return (points,)

    anim = animation.FuncAnimation(
        fig,
        animate,
        init_func=init,
        frames=int(df.shape[0] / (interval * freq)),
        interval=interval * 1000,
        blit=True,
    )

    if save_path.endswith(".gif"):
        writer = animation.PillowWriter(fps=fr)
    elif save_path.endswith(".mp4"):
        writer = animation.FFMpegWriter(fps=fr)
    else:
        raise ValueError(f"Invalid file extension: {save_path}")

    anim.save(save_path, writer=writer)


animate_eyelink(edf, fr=10, save_path=os.path.join(session_dir, "eyelink.mp4"))
animate_eyelink(
    interp, fr=10, save_path=os.path.join(session_dir, "eyelink_interp.mp4")
)

In [ ]:
def animate_markers(df, fr=10, save_path="markers.gif"):
    freq = 1 / df.time.diff().median()
    interval = 1 / fr

    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    (points,) = ax.plot([], [], [], "o")
    ax.set_xlim(df.marker_0_x.min(), df.marker_0_x.max())
    ax.set_ylim(df.marker_0_y.min(), df.marker_0_y.max())
    ax.set_zlim(df.marker_0_z.min(), df.marker_0_z.max())

    def init():
        x, y, z = df.iloc[0][["marker_0_x", "marker_0_y", "marker_0_z"]]
        points.set_data([x], [y])
        points.set_3d_properties([z], "z")
        return (points,)

    def animate(i):
        row = df.iloc[int(i * interval * freq)]
        x, y, z = row[["marker_0_x", "marker_0_y", "marker_0_z"]]
        points.set_data([x], [y])
        points.set_3d_properties([z], "z")
        return (points,)

    anim = animation.FuncAnimation(
        fig,
        animate,
        init_func=init,
        frames=int(df.shape[0] / (interval * freq)),
        interval=interval * 1000,
        blit=True,
    )

    if save_path.endswith(".gif"):
        writer = animation.PillowWriter(fps=fr)
    elif save_path.endswith(".mp4"):
        writer = animation.FFMpegWriter(fps=fr)
    else:
        raise ValueError(f"Invalid file extension: {save_path}")

    anim.save(save_path, writer=writer)


animate_markers(mdf, fr=10, save_path=os.path.join(session_dir, "markers.mp4"))
animate_markers(
    interp, fr=10, save_path=os.path.join(session_dir, "markers_interp.mp4")
)